# Exploratory geospatial analysis

Walking through the catchment pipeline step by step on the sample venue data:
load and validate, reproject, buffer, compute features, score, and map. The
production path lives in `src/build_outputs.py`; this notebook is for poking at
the intermediate steps.

In [ ]:
import sys
from pathlib import Path

# Make the src package importable when running from notebooks/.
sys.path.append(str(Path.cwd().parent))

from src import config, load_data, geospatial_utils as geo, scoring
import geopandas as gpd
import matplotlib.pyplot as plt

## 1. Load and validate

`load_venues` enforces required columns and coordinate sanity before anything
spatial happens.

In [ ]:
df = load_data.load_venues(config.SAMPLE_DATA)
gdf = load_data.to_geodataframe(df)
print(gdf.crs)
gdf.head()

## 2. Reproject to a projected CRS

Distances and areas only make sense in a projected CRS. Note how the coordinate
values change from degrees to meters.

In [ ]:
gdf_proj = geo.to_projected(gdf)
print('Geographic:', gdf.geometry.iloc[0])
print('Projected :', gdf_proj.geometry.iloc[0])

## 3. Build catchment buffers

In [ ]:
buffers_25 = geo.create_buffers(gdf_proj, miles=25)
buffers_25[['venue_name', 'buffer_miles', 'buffer_area_sq_mi']].head()

## 4. Proximity features

In [ ]:
df['n_competitors'] = geo.count_competitors_within(gdf_proj, config.COMPETITOR_RADIUS_MILES)
df['dist_to_nearest_competitor_mi'] = [round(d, 2) for d in geo.distance_to_nearest_competitor(gdf_proj)]
df[['venue_name', 'n_competitors', 'dist_to_nearest_competitor_mi']].sort_values('n_competitors', ascending=False).head()

## 5. Opportunity score

In [ ]:
df['opportunity_score'] = scoring.compute_opportunity_score(df)
ranked = df.sort_values('opportunity_score', ascending=False)
ranked[['venue_name', 'metro_population', 'n_competitors', 'dist_to_nearest_competitor_mi', 'opportunity_score']].head(10)

## 6. Quick look at the catchments

In [ ]:
buffers_wgs = buffers_25.to_crs(config.CRS_GEOGRAPHIC)
gdf_scored = gdf.merge(df[['venue_id', 'opportunity_score']], on='venue_id')

fig, ax = plt.subplots(figsize=(11, 8))
buffers_wgs.plot(ax=ax, color='#1b6ec2', alpha=0.12, edgecolor='#1b6ec2', linewidth=0.6)
gdf_scored.plot(ax=ax, column='opportunity_score', cmap='viridis', markersize=55, legend=True)
ax.set_title('25-mile catchment and opportunity score')
plt.show()